# 🐍 Day 13 · Python Basics — Decorators, Docstrings & Human Approval (Tools and HITL)

> **Goal of this notebook:** The `@tool` decorator turns a normal function into something an LLM can call. Learn docstrings and decorators, then pause for human approval.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. Docstrings — describing a function
2. Decorators — wrapping a function
3. Build your own `@tool` registry
4. Letting a (pretend) LLM pick a tool
5. Human-in-the-loop: approve or reject
6. Raising your own errors

## 1. Docstrings
A **docstring** is the text in triple quotes right under `def`. LangChain sends it to the LLM so the model knows **what the tool does**.

In [ ]:
def get_stock_price(symbol: str) -> float:
    """Return the latest stock price for a company symbol like 'AAPL'."""
    prices = {"AAPL": 230.5, "TSLA": 251.0}
    return prices.get(symbol.upper(), 0.0)

print(get_stock_price.__doc__)      # read the docstring
print(get_stock_price("aapl"))

## 2. Decorators
A **decorator** is a function that wraps another function to add extra behaviour. You apply it with `@name` above `def`.

In [ ]:
def log_calls(func):
    def wrapper(*args):
        print(f"📞 calling {func.__name__} with {args}")
        result = func(*args)
        print(f"✅ result: {result}")
        return result
    return wrapper

@log_calls                      # same as: add = log_calls(add)
def add(a, b):
    return a + b

add(2, 3)

## 3. 🔗 Build your own `@tool`
This decorator **registers** each function in a dictionary together with its description — that list is what gets shown to the LLM.

In [ ]:
TOOLS = {}

def tool(func):
    TOOLS[func.__name__] = {"function": func, "description": func.__doc__}
    return func

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@tool
def search_web(query: str) -> str:
    """Search the internet for a query."""
    return f"(pretend search results for '{query}')"

for name, info in TOOLS.items():
    print(f"{name:<12} -> {info['description']}")

## 4. A pretend LLM chooses a tool
A real LLM answers with a **tool call**: the tool name and its arguments (a dictionary).

In [ ]:
tool_call = {"name": "multiply", "args": {"a": 230.5, "b": 50}}   # what an LLM would send

func = TOOLS[tool_call["name"]]["function"]
result = func(**tool_call["args"])      # ** turns the dict into a=..., b=...
print("Cost of 50 shares:", result)

## 5. Human-in-the-loop
Before a **risky** action (sending email, buying shares) the graph *pauses* and asks a human.
> In real code you would use `input("Approve? (yes/no) ")`. Here we set the answer in a variable so the notebook runs by itself.

In [ ]:
def buy_shares(symbol, qty, human_answer):
    print(f"🤖 Agent wants to buy {qty} shares of {symbol}.")
    if human_answer.strip().lower() in ["yes", "y"]:
        return f"✅ Bought {qty} {symbol}"
    return "❌ Cancelled by human"

print(buy_shares("AAPL", 50, human_answer="yes"))
print(buy_shares("TSLA", 1000, human_answer="no"))

## 6. Raising your own errors
`raise` stops the function and reports a problem. `try / except` catches it.

In [ ]:
def withdraw(balance, amount):
    if amount > balance:
        raise ValueError("Not enough money!")
    return balance - amount

try:
    print(withdraw(100, 30))
    print(withdraw(100, 500))
except ValueError as e:
    print("Error:", e)

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Write a function with a docstring and print the docstring with `__doc__`.
2. Register a new `@tool` called `to_upper(text)` and check that it appears in `TOOLS`.
3. Change `human_answer` to `"Y"` — does it still approve? Why?

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
@tool
def to_upper(text: str) -> str:
    """Make text uppercase."""
    return text.upper()

print("to_upper" in TOOLS, TOOLS["to_upper"]["description"])
# "Y" works because we call .lower() before comparing
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Docstring | `'''Explain what it does.'''` |
| Read docstring | `func.__doc__` |
| Decorator | `@log_calls above def` |
| Function name | `func.__name__` |
| Dict as arguments | `func(**args)` |
| Raise error | `raise ValueError('msg')` |

## ➡️ What's next?
You now know the Python used in **Day_13_LangGraph_Tools_and_Human_in_the_Loop**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀